### write a script that reads the October export, totals spend by category, and tells us which lines it couldn't read, so we can chase those up? 

challenge -> malformed lines

1. **Field count**: exactly 4 fields after splitting on `;`. (Think about what `split(";")` returns for a line that uses `|` instead.)
2. **Date**: exactly 10 characters, a `-` at positions 4 and 7, and digits everywhere else (slicing + `isdigit`).
3. **Merchant**: not blank after cleaning.
4. **Type**: `debit` or `refund` after cleaning.
5. **Amount**: after removing `$`, `,`, spaces and a single leading `-`, what's left must be digits with at most one decimal point. `"N/A"`, `"1O.49"` (letter O) and `""` must all fail. This is the same idea as the `looks_numeric` exercise in the 2.2.2 practice notebook.

must appear in the report with 
- line number
- reason
- raw text

When you're done, deliberately break three more lines of `RAW_EXPORT` in new ways (e.g. an amount of `$12.3.4`, a date of `2026-1-05`, a lone `;;;`) and check your script reports each one correctly, then restore the original data before your final commit.

In [5]:
RAW_EXPORT = """# Corporate card export - October 2026 - generated by LegacyBank v2.3
date;merchant;amount;type
2026-10-01;  FRESHMART GROCERY ;$84.12;debit
2026-10-01;uber trip;$18.40;DEBIT
2026-10-02;Slack Technologies;$1,234.50;debit
2026-10-02;Bean There Cafe;$5.25;debit
2026-10-03;freshmart grocery;$62.07;Debit
2026-10-03;Shell Gas Station;$71.30;debit
2026-10-04;  Uber   Trip;$22.15 ;debit
2026-10-04;GitHub Inc;$21.00;debit

2026-10-05;Bean There Cafe;$4.95;debit
2026-10-05;   ;$13.00;debit
2026-10-06;Noodle House;$38.60;debit
2026-10-06;FreshMart Grocery;-$12.40;refund
2026-10-07;Adobe Systems;$54.99;debit
2026-10-07;Shell gas station;$65.80; debit
2026-10-08;Noodle House;$41.25;debit
2026-10-08;Bean There Cafe;N/A;debit
2026-10-09;Corner Market;$9.99;debit
2026/10/09;UBER TRIP;$15.75;debit
2026-10-10;Office Depot;$143.20;debit
2026-10-10;GitHub Inc;$21.00
2026-10-11;Pizza Palace;$27.50;debit
2026-10-11;Gastown Cafe;$16.80;debit
2026-10-12;Shell Gas Station;$58.10;debit
2026-10-12;Corner Market;$1O.49;debit
2026-10-13;Adobe Systems;$54.99;REFUND
2026-10-13;Transit Pass Kiosk;$156.00;debit
2026-10-14;Pizza Palace;$31.80;debit
2026-10-14;freshmart grocery;$97.45;debit

2026-10-15;Office Depot;$12.35;debit
2026-10-15|Noodle House|$22.10|debit
2026-10-16;Zoom Video;$19.99;debit
2026-10-16;Bean There Cafe;$5.25;chargeback
2026-10-17;Corner Market;$-8.00;refund
2026-10-17;Uber Trip;$19.60;debit"""

In [40]:
# cell cleaning
# skip non-data lines, split by ;, skip rows without 4 ';', skip wrong date, skip blank merchants, skip type not debit or refund, remove $, ',', 


# initialization variables for cell cleaning and validation (questions 1-4)
valid = 0
lines_read = 0
skipped = 0
malformed = 0

# initialization variables for accumulation values (question 7)
    # running counts
        # categories
software_items = 0             # running count for number of items in 'software' category
groceries_items = 0            # running count for number of items in 'groceries' category
dining_items = 0               # running count for number of items in 'dining' category
transport_items = 0            # running count for number of items in 'transport' category
other_items = 0                # running count for number of items in 'other' category

        # types
num_refunds = 0                # running count for number of valid refunds

    # running totals
total_software = 0             # running total value for items in 'software' category
total_groceries = 0            # running total value for items in 'groceries' category
total_dining = 0               # running total value for items in 'dining' category
total_transport = 0            # running total value for items in 'transport' category
total_other = 0                # running total value for items in 'other' category

grand_total = 0                # running total value for all valid transactions
refund_total = 0               # running total value for all valid refunds

    # other tracking variables
largest_charge = 0             # tracks the largest charge within the valid transactions

# list variables for reporting
malformed_report = []


# TASKS 1 THROUGH 7


# question 1: 'walk the lines'
    # looping over each line in the raw export data
for line_no, line in enumerate(RAW_EXPORT.splitlines(), start=1):
    lines_read += 1

# question 2: 'skip non-data lines' --> skips title and header lines along with blank lines. tracks no. of skipped lines
    if not line.strip() or line.strip().startswith('#') or line.lower().startswith('date;'):
        skipped += 1
        continue

# question 3: 'split and clean'
    fields = line.split(';')                               # split each line on ';' --> track malformed lines for report
    if len(fields) != 4:
        malformed += 1
        malformed_report.append(f"line {line_no}: wrong field count: {line}")
        continue

    # clean each field (date, merchant, type, amount)
        # date: strip spaces
    date = fields[0].strip()

        # merchants
    merchant = fields[1].strip()                          # strip spaces
    while '  ' in merchant:
        merchant = merchant.replace('  ', ' ')            # collapse interal runs of space to one space
    merchant = merchant.title()                           # title case

        # types
    tx_type = fields[3].lower().strip()                   # strip and lowercase

        # amounts
    amount = fields[2].replace('$','').replace(',','').replace(' ','').strip()      # strip and remove '$', ',' and spaces
    is_negative = amount.startswith('-')
    if is_negative == True:
        amount = amount[1:]         # strips the leading negative sign only for negative amounts by splitting at the first position

    
# question 4: validate --> challenge component. all invalid lines are tracked as 'malformed' and added to the report
    # date validation
        # date length exactly 10, '-' at positions 4 and 7, only digits outside of hyphens
    if len(date) != 10 or date[4] != '-' or date[7] != '-' or date.replace('-','').isdigit() == False:
        malformed += 1
        malformed_report.append(f"line {line_no}: invalid date format: {line}")
        continue

    # merchant validation
        # no blank merchants after cleaning
    if merchant == '':
        malformed += 1
        malformed_report.append(f"line {line_no}: blank merchant: {line}")
        continue

    # type
        # only types: 'debit' and 'refund'
    if tx_type != 'debit' and tx_type != 'refund':
        malformed += 1
        malformed_report.append(f"line {line_no}: invalid type format: {line}")
        continue

    # amount
        # removes amounts more than 1 decimal and non-digit values within the amount
    if not amount.replace('.','',1).isdigit():
        malformed += 1
        malformed_report.append(f"line {line_no}: invalid amount - non-numeric format: {line}")
        continue


# question 5: 'convert the amount' --> amount value converted to a float. refunds recieve a negative value (negative returned back)
    value = float(amount)
    if tx_type == 'refund':
        value = -value




# question 6: categorization --> based on table in project outline
    # assigns a category variable based on string values within the merchant name using if/elif/else loop
    if 'slack' in merchant.lower() or 'github' in merchant.lower() or 'adobe' in merchant.lower() or 'zoom' in merchant.lower():
        category = "Software"
    elif 'freshmart' in merchant.lower() or 'market' in merchant.lower():
        category = "Groceries"
    elif 'cafe' in merchant.lower() or 'noodle' in merchant.lower() or 'pizza' in merchant.lower():
        category = "Dining"          # 'gastown cafe ALWAYS has to come before the 'transport' assignment to avoid mislabelling
    elif 'uber' in merchant.lower() or 'gas' in merchant.lower() or 'transit' in merchant.lower():
        category = "Transport"
    else:
        category = "Other"


# question 7: accumulate
    # Running total of each category
    if category == "Software":
        total_software += value
    elif category == "Groceries":
        total_groceries += value
    elif category == "Dining":
        total_dining += value
    elif category == "Transport":
        total_transport += value
    else:
        total_other += value
    
    # Count of items in each category
    match category:
        case "Software":
            software_items += 1
        case "Groceries":
            groceries_items += 1
        case "Dining":
            dining_items += 1
        case "Transport":
            transport_items += 1
        case "Other":
            other_items += 1
 
    # Count of valid transactions 
    valid += 1
    
    # net grand total
    grand_total += value
    
    # Total Count of refunds and the total amount
    if tx_type == "refund":
        num_refunds += 1
        refund_total += -value
    
    # Single largest charge with merchant and date
    if value > largest_charge:
        largest_charge = value
        largest_charge_merchant = merchant
        largest_charge_date = date
    
print(f"software stats: {software_items} items, ${total_software}")
print(f"grocery stats: {groceries_items} items, ${total_groceries}")
print(f"dining stats: {dining_items} items, ${total_dining}")
print(f"transport stats: {transport_items} items, ${total_transport}")
print(f"other stats: {other_items} items, ${total_other}\n")
print(f"The grand running total comes out to ${grand_total}, with {valid} valid transactions.\n")
print(f"There were a total of {num_refunds} refunds totalling ${refund_total}. largest charge: {largest_charge}, merchant: {largest_charge_merchant}, date of transaction: {largest_charge_date}")



software stats: 5 items, $1275.49
grocery stats: 6 items, $233.23000000000002
dining stats: 7 items, $166.15
transport stats: 7 items, $411.35
other stats: 2 items, $155.54999999999998

The grand running total comes out to $2241.7699999999995, with 27 valid transactions.

There were a total of 3 refunds totalling $75.39. largest charge: 1234.5, merchant: Slack Technologies, date of transaction: 2026-10-02
